In [26]:
import sys
import numpy as np
from astropy import units as u
from scipy.integrate import quad
sys.path.append("..")
import src.cosmology as cosmo
from astropy.cosmology import FlatLambdaCDM
import emcee
import matplotlib.pyplot as plt
import corner

In [2]:
results, err = quad(cosmo.age_integrand, 0, 20, args=(0.3,))
print(results)

print((977.8/13.5) * quad(cosmo.age_integrand, 0, 11, args=(0.3,))[0] * u.km/u.s/u.Mpc)
print((977.8/13.5) * quad(cosmo.age_integrand, 0, 20, args=(0.3,))[0] * u.km/u.s/u.Mpc)
print((977.8/13.5) * quad(cosmo.age_integrand, 0, 30, args=(0.3,))[0] * u.km/u.s/u.Mpc)

0.9514519909002362
67.70907331754697 km / (Mpc s)
68.91331531127786 km / (Mpc s)
69.3186019734332 km / (Mpc s)


In [3]:
result = FlatLambdaCDM(H0=70 * u.km/u.s/u.Mpc, Om0=0.3)
print(result.lookback_time(20))


13.290319371499207 Gyr


In [6]:

def L(age, sigma_age, H0, Om0, zf):
    likelihood = -0.5 * (age - FlatLambdaCDM(H0, Om0).lookback_time(zf).value )**2 / sigma_age **2
    return likelihood
    ## This result is already ln of the likelihood (Gaussian)

print (L(13.5, 0.5, 68.9, 0.3, 20))
print (L(13.5, 0.5, 80, 0.3, 20))

def L_prior(H0, Om0, zf):
    if H0 < 50 or H0 > 100 or zf < 11 or zf > 30:
        return -np.inf
    likelihood = -0.5 * ((Om0 - 0.30) / 0.02)**2
    return likelihood

L_prior(70, 0.32, 20)

def L_posterior(vector):
    H0, Om0, zf = vector
    if np.isneginf(L_prior(H0, Om0, zf)):
        return -np.inf
    return L(13.5, 0.5, H0, Om0, zf) + L_prior(H0, Om0, zf)


-1.2515390067189833e-05
-7.001061597472032


In [22]:
ndim = 3
nwalkers = 32

rng = np.random.default_rng(0)

weights = [1, 0.005, 1] #H0, Om0, zf respectively

input = np.array([69, 0.30, 20]) + weights * rng.standard_normal((nwalkers, ndim))

sampler = emcee.EnsembleSampler(nwalkers, ndim, L_posterior)

sampler.run_mcmc(input, 3000, progress = True)

samples = sampler.get_chain(flat=True) # flat = True collapses into single array posterior sample

H0_samples = samples[:, 0]

100%|██████████| 3000/3000 [09:03<00:00,  5.52it/s]


In [ ]:
print(np.percentile(H0_samples, [16, 50, 84]), sampler.acceptance_fraction.mean())
print(sampler.get_autocorr_time(quiet = True))

for i in range(3):
    plt.figure()
    plt.plot(sampler.get_chain()[300:, 1, i], alpha = 0.3)

In [ ]:
figure = corner.corner(sampler.get_chain(discard = 300, flat = True), bins=50, smooth = 1.0, smooth1d = 1.0, labels = (r"$H_0$", r"$\Omega_m$", r"$z_f$"))